In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
#converting api endpoints to data frames
owners_url = "https://www.fema.gov/api/open/v2/HousingAssistanceOwners"
renters_url = "https://www.fema.gov/api/open/v2/HousingAssistanceRenters"

def fetch_all(url, params):
    all_data = []
    skip = 0
    key = url.split("/")[-1]  #gets "HousingAssistanceOwners" or "HousingAssistanceRenters"
    
    while True:
        params["$skip"] = skip
        response = requests.get(url, params=params).json()[key]
        
        if not response:
            break
            
        all_data.extend(response)
        
        if len(response) < 1000:  #last page
            break
            
        skip += 1000
    
    return all_data

params = {
    "$filter": "disasterNumber eq 4827",
    "$select": "state,validRegistrations,approvedForFemaAssistance,totalApprovedIhpAmount",
    "$top": 1000,
    "$format": "json"
}

owners_df = pd.DataFrame(fetch_all(owners_url, params.copy()))
renters_df = pd.DataFrame(fetch_all(renters_url, params.copy()))

print(len(owners_df), len(renters_df))

1409 1457


In [3]:
print(owners_df.dtypes)
print()
print(owners_df.head(2))
print()
print(owners_df.isnull().sum())

state                         object
validRegistrations             int64
approvedForFemaAssistance      int64
totalApprovedIhpAmount       float64
dtype: object

  state  validRegistrations  approvedForFemaAssistance  totalApprovedIhpAmount
0    NC                   2                          2                 8220.63
1    NC                   1                          1                  750.00

state                        0
validRegistrations           0
approvedForFemaAssistance    0
totalApprovedIhpAmount       0
dtype: int64


In [4]:
renters_df['state'].nunique()
print(renters_df['state'].unique())

['NC' 'SC']


In [5]:
print(owners_df[['validRegistrations', 'approvedForFemaAssistance', 'totalApprovedIhpAmount']].describe())
print()
print(renters_df[['validRegistrations', 'approvedForFemaAssistance', 'totalApprovedIhpAmount']].describe())

       validRegistrations  approvedForFemaAssistance  totalApprovedIhpAmount
count         1409.000000                1409.000000            1.409000e+03
mean           120.065295                  73.718950            3.325276e+05
std            543.901595                 388.934161            1.865622e+06
min              1.000000                   0.000000            0.000000e+00
25%              1.000000                   0.000000            0.000000e+00
50%              2.000000                   1.000000            3.000000e+02
75%             14.000000                   3.000000            1.312248e+04
max           7354.000000                5555.000000            3.206580e+07

       validRegistrations  approvedForFemaAssistance  totalApprovedIhpAmount
count         1457.000000                1457.000000            1.457000e+03
mean            77.470830                  39.155113            6.578854e+04
std            422.839535                 249.493642            3.829651e+0